# pixel-trust v3: a second, real generator (GenWarp, NeurIPS 2024)

Same KITTI Stereo 2015 protocol as v2, but the view is synthesized by **GenWarp** (Seo et al., NeurIPS 2024), which regenerates the whole target view from the source image, its depth prior (Depth Anything V2 aligned to 10% LiDAR) and the stereo pose. GenWarp works on 512x512 squares: each frame is covered by four overlapping 368x368 crops that are stitched column-wise.

Every trust signal and every published baseline (sample variance, trajectory variance, semantic entropy, VGGT cross-view confidence) is computed and calibrated exactly as in v2.

Settings: Accelerator **GPU T4 x2**, Internet **on**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess, sys
REPO = "https://github.com/Rahul-Verma97/pixel-trust.git"
if not os.path.isdir("/kaggle/working/pixel-trust"):
    !git clone -q {REPO} /kaggle/working/pixel-trust
!pip install -q "diffusers>=0.27" "transformers>=4.45" accelerate omegaconf einops roma jaxtyping
os.chdir("/kaggle/working/pixel-trust")
!git log -1 --format='%h %s'

In [ ]:
# KITTI Stereo 2015: use an attached Kaggle copy if there is one, else download the official zip
sys.path.insert(0, "/kaggle/working/pixel-trust")
from pixel_trust.data import find_kitti_root, download_kitti
root = find_kitti_root(["/kaggle/input", "/kaggle/working/kitti"])
if root is None:
    root = download_kitti("/kaggle/working/kitti")
    root = find_kitti_root(["/kaggle/working/kitti"])
print("KITTI root:", root, "| pairs:", len(os.listdir(os.path.join(root, "training", "disp_occ_0"))))

In [ ]:
# Smoke test: 2 images on GPU 0, to confirm every model and baseline loads before the full run
import glob, numpy as np
!CUDA_VISIBLE_DEVICES=0 python run_generate.py --root {root} --out /kaggle/working/smoke --limit 2 --k 4 --steps 20 --gen genwarp 2>&1 | tail -20
for f in sorted(glob.glob('/kaggle/working/smoke/feats/*.npz'))[:2]:
    z = np.load(f); print(f.split('/')[-1], sorted(k for k in z.files if k.startswith('f_')))

In [ ]:
# Stage 1: generate views + trust features, one process per GPU
import time
OUT = "/kaggle/working/out"
import torch
ngpu = max(1, torch.cuda.device_count())
procs = []
for g in range(ngpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
    log = open(f"/kaggle/working/gen_shard{g}.log", "w")
    procs.append(subprocess.Popen([sys.executable, "run_generate.py", "--root", root, "--out", OUT,
                                   "--shard", str(g), "--nshards", str(ngpu), "--k", "4", "--steps", "20", "--gen", "genwarp"],
                                  stdout=log, stderr=subprocess.STDOUT, env=env))
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(120)
    tails = [open(f"/kaggle/working/gen_shard{g}.log").read().strip().splitlines()[-1:] for g in range(ngpu)]
    print(f"{(time.time()-t0)/60:5.1f} min", tails, flush=True)
print("exit codes:", [p.returncode for p in procs])
for g in range(ngpu):
    print(open(f"/kaggle/working/gen_shard{g}.log").read()[-3000:])

In [ ]:
# Stage 2: analysis, figures and report
!python run_analysis.py --out /kaggle/working/out
print(open("/kaggle/working/out/REPORT.md").read())

In [ ]:
from IPython.display import Image, display
for f in ["fig_sota", "fig_examples", "fig_sparsification", "fig_conformal", "fig_samples"]:
    p = f"/kaggle/working/out/figures/{f}.png"
    if os.path.exists(p):
        print(f); display(Image(p))

In [ ]:
# Package the small outputs (report, numbers, figures, example images); the per-pixel features stay out
!cd /kaggle/working/out && zip -q -r /kaggle/working/pixel_trust_genwarp_results.zip REPORT.md results.json figures png log_shard*.jsonl
!ls -la /kaggle/working/pixel_trust_genwarp_results.zip